# 01 — Analisi Preliminare

Notebook dedicato al caricamento del dataset, pulizia delle colonne e prima analisi esplorativa.

**Sprint 1 | Settimana 1**

In [1]:
# Librerie principali
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Impostazioni grafici
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print("Librerie importate correttamente")

Librerie importate correttamente


## US-01 · Caricamento e Prima Ispezione del Dataset

In [4]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("aai510-group1/telco-customer-churn")


df = pd.concat([
    dataset['train'].to_pandas(),
    dataset['validation'].to_pandas(),
    dataset['test'].to_pandas()
], ignore_index=True)

# Salva il dataset completo in data/raw/
df.to_csv("C:\\Users\\david\\OneDrive\\Desktop\\Project Dev\\telco-churn-ml\\data\\raw\\Telco_churn.csv", index=False)
print("Dataset salvato correttamente")

print(df.shape)  # (7043, 52)

Dataset salvato correttamente
(7043, 52)


In [5]:
# Verifica dimensioni
print(f"Righe: {df.shape[0]}")
print(f"Colonne: {df.shape[1]}")

# Tipi di dato per ogni colonna
print("\nTipi di dato:")
print(df.dtypes)

Righe: 7043
Colonne: 52

Tipi di dato:
Age                                    int64
Avg Monthly GB Download                int64
Avg Monthly Long Distance Charges    float64
Churn                                  int64
Churn Category                           str
Churn Reason                             str
Churn Score                            int64
City                                     str
CLTV                                   int64
Contract                                 str
Country                                  str
Customer ID                              str
Customer Status                          str
Dependents                             int64
Device Protection Plan                 int64
Gender                                   str
Internet Service                       int64
Internet Type                            str
Lat Long                                 str
Latitude                             float64
Longitude                            float64
Married         

In [6]:
# Prime 5 righe
print("Prime 5 righe:")
df.head()

Prime 5 righe:


,Age,Avg Monthly GB Download,Avg Monthly Long Distance Charges,Churn,Churn Category,Churn Reason,Churn Score,City,CLTV,Contract,...,Streaming TV,Tenure in Months,Total Charges,Total Extra Data Charges,Total Long Distance Charges,Total Refunds,Total Revenue,Under 30,Unlimited Data,Zip Code
0,72,4,19.44,0,NaN,NaN,51,San Mateo,4849,Two Year,...,0,25,2191.15,0,486.00,0.0,2677.15,0,1,94403
1,27,59,45.62,0,NaN,NaN,27,Sutter Creek,3715,Month-to-Month,...,1,35,3418.20,0,1596.70,0.0,5014.90,1,1,95685
2,59,0,16.07,0,NaN,NaN,59,Santa Cruz,5092,Month-to-Month,...,0,46,851.20,0,739.22,0.0,1590.42,0,0,95064
3,25,27,0.00,0,NaN,NaN,49,Brea,2068,One Year,...,0,27,1246.40,30,0.00,0.0,1276.40,1,0,92823
4,31,21,17.22,1,Dissatisfaction,Network reliability,88,San Jose,4026,One Year,...,0,58,3563.80,0,998.76,0.0,4562.56,0,1,95117


In [7]:
df.tail()

,Age,Avg Monthly GB Download,Avg Monthly Long Distance Charges,Churn,Churn Category,Churn Reason,Churn Score,City,CLTV,Contract,...,Streaming TV,Tenure in Months,Total Charges,Total Extra Data Charges,Total Long Distance Charges,Total Refunds,Total Revenue,Under 30,Unlimited Data,Zip Code
7038,33,12,10.67,0,NaN,NaN,53,Clayton,4847,Month-to-Month,...,1,29,2878.75,0,309.43,0.0,3188.18,0,1,94517
7039,36,30,30.44,0,NaN,NaN,45,Nicolaus,5470,Two Year,...,1,59,5655.45,0,1795.96,0.0,7451.41,0,1,95659
7040,57,24,4.51,0,NaN,NaN,74,Walnut Creek,4471,Two Year,...,1,61,5175.30,0,275.11,0.0,5450.41,0,1,94596
7041,35,24,47.06,0,NaN,NaN,70,Valyermo,5445,Two Year,...,0,30,1653.85,0,1411.80,0.0,3065.65,0,1,93563
7042,54,24,12.85,0,NaN,NaN,70,Rancho Cucamonga,5065,Two Year,...,1,72,7998.80,70,925.20,0.0,8994.00,0,0,91701


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 52 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Age                                7043 non-null   int64  
 1   Avg Monthly GB Download            7043 non-null   int64  
 2   Avg Monthly Long Distance Charges  7043 non-null   float64
 3   Churn                              7043 non-null   int64  
 4   Churn Category                     1869 non-null   str    
 5   Churn Reason                       1869 non-null   str    
 6   Churn Score                        7043 non-null   int64  
 7   City                               7043 non-null   str    
 8   CLTV                               7043 non-null   int64  
 9   Contract                           7043 non-null   str    
 10  Country                            7043 non-null   str    
 11  Customer ID                        7043 non-null   str    
 12  Cus

## US-02 · Rimozione delle Colonne Leaky e Inutili

# Cos'è il Data Leakage?
Il data leakage si verifica quando il modello riceve informazioni che nella realtà
non sarebbero disponibili al momento della predizione.

**Esempio concreto da questo dataset**: immagina di essere un analista del 
Dipartimento Retention il lunedì mattina. Hai davanti i dati di un cliente 
ancora attivo. Puoi vedere la sua età, il tipo di contratto, la spesa mensile. 
Non puoi vedere il motivo per cui se ne andrà (`Churn Reason`) o la categoria 
del suo abbandono (`Churn Category`) — perché non è ancora andato via.
Usare queste informazioni nel modello sarebbe come consegnare le risposte 
durante un esame: il modello sembrerebbe perfetto, ma nella realtà fallirebbe.

In [9]:
print(df.nunique().sort_values())

Country                                 1
Quarter                                 1
State                                   1
Churn                                   2
Dependents                              2
Multiple Lines                          2
Gender                                  2
Device Protection Plan                  2
Online Backup                           2
Partner                                 2
Paperless Billing                       2
Online Security                         2
Phone Service                           2
Premium Tech Support                    2
Married                                 2
Internet Service                        2
Streaming Movies                        2
Streaming Music                         2
Referred a Friend                       2
Senior Citizen                          2
Under 30                                2
Unlimited Data                          2
Streaming TV                            2
Internet Type                     

### Premessa metodologica
Prima di procedere con qualsiasi analisi, il team ha identificato e rimosso 
le colonne che non possono essere utilizzate come feature predittive. Questa 
fase è critica per garantire che il modello rifletta fedelmente le informazioni 
disponibili al momento della predizione nella realtà operativa aziendale.

### Colonne rimosse per Data Leakage
Le seguenti colonne contengono informazioni disponibili solo *dopo* che il cliente 
ha già abbandonato il servizio. Includerle nel modello invaliderebbe completamente 
i risultati, in quanto il modello "conoscerebbe la risposta in anticipo".

| Colonna | Motivazione |
|---|---|
| Churn Score | Probabilità di abbandono già calcolata da sistema esterno |
| Churn Category | Categoria del motivo di abbandono — disponibile solo post-abbandono |
| Churn Reason | Motivo specifico dell'abbandono — disponibile solo post-abbandono |
| Customer Status | Stato finale del cliente — equivalente al target |

### Colonne rimosse per assenza di valore predittivo
| Colonna | Motivazione |
|---|---|
| Customer ID | Identificatore univoco del cliente — non porta informazione predittiva |
| Country | Valore costante (United States) — nessuna varianza nel dataset |
| State | Valore costante (California) — nessuna varianza nel dataset |
| Quarter | Valore costante (Q3) — nessuna varianza nel dataset |
| Lat Long | Colonna ridondante — Latitude e Longitude già presenti separatamente |
| Under 30 | Ridondante con la colonna Age, già inclusa nel dataset |

In [10]:
colonne_leaky = [
    "Churn Score",      
    "Churn Category",   
    "Churn Reason",     
    "Customer Status"   
]

df = df.drop(columns=colonne_leaky)
print(f"Colonne rimosse per data leakage: {len(colonne_leaky)}")
print(f"Shape dopo rimozione leaky: {df.shape}")

Colonne rimosse per data leakage: 4
Shape dopo rimozione leaky: (7043, 48)


In [11]:
colonne_inutili = [
    "Customer ID",  
    "Country",      
    "State",        
    "Quarter",      
    "Lat Long",     
    "Under 30"      
]

df = df.drop(columns=colonne_inutili)
print(f"Colonne rimosse per assenza di valore predittivo: {len(colonne_inutili)}")
print(f"Shape dopo rimozione colonne inutili: {df.shape}")

Colonne rimosse per assenza di valore predittivo: 6
Shape dopo rimozione colonne inutili: (7043, 42)


### Decisione sulle colonne "grigie"

Dopo valutazione del team, abbiamo deciso di procedere come segue:

| Colonna | Decisione | Motivazione |
|---|---|---|
| Satisfaction Score | Rimossa | Informazione raccolta prevalentemente dopo l'abbandono del cliente — parzialmente leaky |
| CLTV | Mantenuta | Valore economico calcolato a priori su tutti i clienti attivi — feature legittima e informativa |
| City | Rimossa | 1106 categorie uniche — introdurrebbe rumore nel modello senza apportare valore predittivo |
| Zip Code | Rimossa | Ridondante con Population e con troppi valori unici per essere gestita direttamente |
| Latitude | Rimossa | Coordinata geografica non interpretabile direttamente dal modello senza elaborazioni aggiuntive |
| Longitude | Rimossa | Coordinata geografica non interpretabile direttamente dal modello senza elaborazioni aggiuntive |
| Population | Mantenuta | Proxy della densità demografica della zona — fornisce informazioni sul contesto socioeconomico del cliente |

In [12]:
colonne_grigie = [
    "Satisfaction Score",
    "City",              
    "Zip Code",          
    "Latitude",          
    "Longitude"          
]

df = df.drop(columns=colonne_grigie)
print(f"Colonne rimosse dopo valutazione: {len(colonne_grigie)}")
print(f"Shape dopo rimozione colonne grigie: {df.shape}")

Colonne rimosse dopo valutazione: 5
Shape dopo rimozione colonne grigie: (7043, 37)


### Separazione della variabile target

La colonna `Churn` rappresenta la variabile target del progetto:
- `0` → il cliente è rimasto
- `1` → il cliente ha abbandonato il servizio

Viene separata dalle feature per essere utilizzata come variabile dipendente 
nella fase di modellazione.

In [13]:
y = df["Churn"]
X = df.drop(columns=["Churn"])

print(f"Feature (X): {X.shape}")
print(f"Target (y): {y.shape}")
print(f"\nDistribuzione target:")
print(y.value_counts())
print(f"Percentuale churn: {y.mean()*100:.1f}%")

Feature (X): (7043, 36)
Target (y): (7043,)

Distribuzione target:
Churn
0    5174
1    1869
Name: count, dtype: int64
Percentuale churn: 26.5%


### Riepilogo della pulizia

| Fase | Colonne rimosse | Shape risultante |
|---|---|---|
| Dataset originale | — | (7043, 52) |
| Rimozione leaky | 4 | (7043, 48) |
| Rimozione inutili | 6 | (7043, 42) |
| Rimozione colonne grigie | 5 | (7043, 37) |
| Separazione target | 1 | X: (7043, 36) |

**Nota sul bilanciamento delle classi**: il dataset presenta uno sbilanciamento
con il 26.5% di clienti churned. Per questa ragione nelle fasi successive 
verrà utilizzato il **F1-Score** come metrica principale di valutazione, 
e non la semplice Accuracy.

# US-03 · Rinomina delle Variabili

## Per prima cosa andiamo a visionare i nomi delle colonne

In [16]:
print(df.columns.tolist())

['Age', 'Avg Monthly GB Download', 'Avg Monthly Long Distance Charges', 'Churn', 'CLTV', 'Contract', 'Dependents', 'Device Protection Plan', 'Gender', 'Internet Service', 'Internet Type', 'Married', 'Monthly Charge', 'Multiple Lines', 'Number of Dependents', 'Number of Referrals', 'Offer', 'Online Backup', 'Online Security', 'Paperless Billing', 'Partner', 'Payment Method', 'Phone Service', 'Population', 'Premium Tech Support', 'Referred a Friend', 'Senior Citizen', 'Streaming Movies', 'Streaming Music', 'Streaming TV', 'Tenure in Months', 'Total Charges', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Total Refunds', 'Total Revenue', 'Unlimited Data']


## Prepariamo la lista dei nuovi nomi in snake case ed avviamo la rinomina

In [17]:
df = df.rename(columns={
    'Age': 'age',
    'Avg Monthly GB Download': 'avg_monthly_gb_download',
    'Avg Monthly Long Distance Charges': 'avg_monthly_ld_charges',
    'Churn': 'churn',
    'CLTV': 'cltv',
    'Contract': 'contract',
    'Dependents': 'dependents',
    'Device Protection Plan': 'device_protection_plan',
    'Gender': 'gender',
    'Internet Service': 'internet_service',
    'Internet Type': 'internet_type',
    'Married': 'married',
    'Monthly Charge': 'monthly_charge',
    'Multiple Lines': 'multiple_lines',
    'Number of Dependents': 'num_dependents',
    'Number of Referrals': 'num_referrals',
    'Offer': 'offer',
    'Online Backup': 'online_backup',
    'Online Security': 'online_security',
    'Paperless Billing': 'paperless_billing',
    'Partner': 'partner',
    'Payment Method': 'payment_method',
    'Phone Service': 'phone_service',
    'Population': 'population',
    'Premium Tech Support': 'premium_tech_support',
    'Referred a Friend': 'referred_a_friend',
    'Senior Citizen': 'senior_citizen',
    'Streaming Movies': 'streaming_movies',
    'Streaming Music': 'streaming_music',
    'Streaming TV': 'streaming_tv',
    'Tenure in Months': 'tenure_months',
    'Total Charges': 'total_charges',
    'Total Extra Data Charges': 'total_extra_data_charges',
    'Total Long Distance Charges': 'total_ld_charges',
    'Total Refunds': 'total_refunds',
    'Total Revenue': 'total_revenue',
    'Unlimited Data': 'unlimited_data',
})

print(df.columns.tolist())

['age', 'avg_monthly_gb_download', 'avg_monthly_ld_charges', 'churn', 'cltv', 'contract', 'dependents', 'device_protection_plan', 'gender', 'internet_service', 'internet_type', 'married', 'monthly_charge', 'multiple_lines', 'num_dependents', 'num_referrals', 'offer', 'online_backup', 'online_security', 'paperless_billing', 'partner', 'payment_method', 'phone_service', 'population', 'premium_tech_support', 'referred_a_friend', 'senior_citizen', 'streaming_movies', 'streaming_music', 'streaming_tv', 'tenure_months', 'total_charges', 'total_extra_data_charges', 'total_ld_charges', 'total_refunds', 'total_revenue', 'unlimited_data']


## Andiamo a verificare che non siano presenti maiuscole o spazi così che tutto risulti in snake_case

In [18]:
assert all(' ' not in col for col in df.columns), "Trovati spazi nei nomi colonna!"
assert all(col == col.lower() for col in df.columns), "Trovate maiuscole nei nomi colonna!"
print("✅ Tutti i nomi colonna sono validi")

✅ Tutti i nomi colonna sono validi
